# KG1 V1244 CoT-safe — TREINO vigiado (Claude)
Rota A. **Run all.** Pré-req: Colab **A100 80GB** + Secret `HF_KEY` (escrita).
- Fixa `torch==2.10` (cu126) p/ casar o **wheel pronto do mamba** (~10s). GPU-guard aborta se VRAM < 70GB.
- SHA dos dados pinado (versão LF = a do Colab) p/ passar o gate de integridade.
- `MODE='SMOKE'` (ensaio) ou `MODE='REAL'` (160 steps). Juíz de score = Notebook B (full947).

## 🧭 Como ler os logs (estilo *Use a Cabeça*)
Cada etapa imprime `[KG1-TEACH][ESTÁGIO][STATUS]`: **O que é / Por que importa / Como ler / Números-chave / Próxima ação**.
Status: `RUNNING/OK` 🟢 · `WATCH` 🟡 · `STOP/ABORT` 🔴 (watchdog matou).
Travamento: `KG1_WRAPPER_HEARTBEAT` a cada 45s (`last_output_age_s` crescendo = travando). Tudo sobe pro HF.
> 💡 `TRAIN_PULSE`=batida do coração. `SCORE_TRAJECTORY`=teacher-forced (não é score real; juíz=Notebook B).

In [ ]:
import os, subprocess, sys, glob
print('[1/5] clone repo branch', flush=True)
subprocess.run(['git','clone','--depth','1','--branch','claude/v1244-cot-safe','https://github.com/FELIPEACASTRO/KG1-NVIDIA.git','/content/kg1'], check=True)
os.chdir('/content/kg1')
print('[2/5] deps base', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft==0.19.1','accelerate==1.13.0','bitsandbytes','safetensors','huggingface_hub','hf_xet','einops','ninja'], check=False)
print('[3/5] pin torch 2.10 cu126 (~2-3min; p/ casar o wheel do mamba)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.10.0','--index-url','https://download.pytorch.org/whl/cu126'], check=False)
import torch
assert torch.cuda.is_available() and torch.version.cuda, f'torch SEM CUDA apos pin (v={torch.__version__}). Reinicie e rode de novo.'
py=f"cp{sys.version_info.major}{sys.version_info.minor}"
tmm='.'.join(torch.__version__.split('+')[0].split('.')[:2])
cu='cu'+((torch.version.cuda or '12').split('.')[0])
abi='TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
print(f'[env] {py} torch{tmm} {cu} cxx11abi{abi} cuda_ok={torch.cuda.is_available()}', flush=True)
print('[4/5] mamba-ssm 2.3.1 (WHEEL PRONTO ~10s; fallback source)', flush=True)
url=f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{tmm}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
print('  tentando wheel:', url, flush=True)
if subprocess.run([sys.executable,'-m','pip','install','--no-deps',url]).returncode!=0:
    print('  >>> wheel nao casou -> compilando do source (~25min)', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','mamba-ssm==2.3.1'], check=False)
print('[5/5] causal-conv1d 1.6.1 (tenta WHEEL cacheado no HF ~10s; senao compila ~5min)', flush=True)
wtag=f"{py}_torch{tmm}_cu{(torch.version.cuda or 'na').replace('.','')}"
causal_ok=False
try:
    from google.colab import userdata
    for k in ['HF_KEY','HF_TOKEN','HUGGINGFACE_TOKEN']:
        try:
            tv=userdata.get(k)
            if tv: os.environ.setdefault('HF_TOKEN',tv); break
        except Exception: pass
    from huggingface_hub import snapshot_download
    d=snapshot_download('felipesp1983/kg1-wheels', repo_type='dataset', allow_patterns=f'{wtag}/causal*', token=os.environ.get('HF_TOKEN'))
    cw=glob.glob(f'{d}/{wtag}/causal*.whl')
    if cw and subprocess.run([sys.executable,'-m','pip','install','--no-deps']+cw).returncode==0:
        print('  causal via WHEEL cacheado (HF) ~10s', flush=True); causal_ok=True
except Exception as e:
    print('  sem cache causal (', str(e)[:70], ')', flush=True)
if not causal_ok:
    print('  compilando causal do source (~5min)', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','causal-conv1d==1.6.1'], check=False)
try:
    import mamba_ssm, causal_conv1d
    print('IMPORT OK: mamba_ssm + causal_conv1d', flush=True)
except Exception as e:
    raise RuntimeError('FALHA import mamba_ssm/causal_conv1d apos install: '+str(e)[:200]+' -> reinicie o runtime e rode de novo.')
print('DEPS OK', flush=True)

In [ ]:
from google.colab import userdata
import os
for k in ['HF_KEY','HF_TOKEN','HUGGINGFACE_TOKEN']:
    try:
        v=userdata.get(k)
        if v: os.environ['HF_TOKEN']=v; os.environ['HF_KEY']=v; break
    except Exception: pass
assert os.environ.get('HF_TOKEN'), 'Defina HF_KEY no Colab Secrets (com escrita)'
print('HF token OK', flush=True)

In [ ]:
import torch
name=torch.cuda.get_device_name(0)
vram=torch.cuda.get_device_properties(0).total_memory/1e9
print(f'[GPU] {name} | VRAM total={vram:.0f}GB | usada={torch.cuda.memory_allocated(0)/1e9:.1f}GB', flush=True)
assert vram>=70, (f'VRAM={vram:.0f}GB < 70GB. O 30B precisa ~60GB -> esta A100 e 40GB. '
    'RECONECTE p/ A100 80GB (desconecte e reconecte ate vir 80GB). NAO treine em 40GB (OOM).')
print('[GPU] OK: VRAM suficiente p/ o 30B', flush=True)

In [ ]:
import os, time
# ==========================================================================
#  ESCOLHA O MODO  (1 unico knob)
MODE = 'SMOKE'  # COMECE com 'SMOKE'. Troque p/ 'REAL' (160 steps) SO apos o GO do Claude.
# ==========================================================================
os.environ['DATA_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_micro_consolidation_train.jsonl'
os.environ['VAL_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_micro_consolidation_train.jsonl'  # val=train: o eval-set 170 e bare (sem messages); juiz real e o Notebook B
# SHA256 dos dados (versao LF = a que o git/Colab entrega). Faz o gate de integridade PASSAR.
os.environ['EXPECTED_TRAIN_SHA256']='8d90bb2de38ab13a438092036b039469c3b5f73ac0f6fdf1f7e27a0b5bde8c2c'
os.environ['EXPECTED_VAL_SHA256']='8d90bb2de38ab13a438092036b039469c3b5f73ac0f6fdf1f7e27a0b5bde8c2c'  # val=train -> mesmo sha do train
# Dataset v1244 e MICRO (979/170) de proposito -> baixar os gates de tamanho (defaults eram p/ o dataset antigo 8777/720).
os.environ['MIN_TRAIN_EXAMPLES']='500'; os.environ['MIN_VAL_EXAMPLES']='100'
os.environ['MIN_TOKENIZED_TRAIN_EXAMPLES']='500'; os.environ['MIN_TOKENIZED_VAL_EXAMPLES']='100'
os.environ['INIT_ADAPTER_REPO']='felipesp1983/kg1-recovered-v291-v290-checkpoint6-submit086'
os.environ['INIT_ADAPTER_REVISION']='f4134a6d223249d27be2f1c5d94ed59d118d1ce5'
os.environ['REQUIRE_INIT_ADAPTER']='1'
os.environ['LORA_R']='32'; os.environ['LORA_ALPHA']='32'
os.environ['BATCH_SIZE']='32'
os.environ['LEARNING_RATE']='5e-6'; os.environ['FINAL_LEARNING_RATE']='1e-6'
os.environ['BOXED_PAYLOAD_LOSS_WEIGHT']='1.0'
os.environ['REQUIRE_OFFSET_MASK']='1'
os.environ['OUTPUT_REPO']='felipesp1983/kg1-v1244-cot-candidate'
os.environ['UPLOAD_TO_HF']='1'
if MODE=='REAL':
    os.environ['MAX_STEPS']='160'; os.environ['NUM_EPOCHS']='6'
    os.environ['EVAL_EVERY_STEPS']='40'; os.environ['SAVE_EVERY_STEPS']='40'
    os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD']='0'
else:
    os.environ['MAX_STEPS']='8'; os.environ['NUM_EPOCHS']='1'
    os.environ['EVAL_EVERY_STEPS']='4'; os.environ['SAVE_EVERY_STEPS']='4'
    os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD']='1'
print('MODE=',MODE,'| MAX_STEPS=',os.environ['MAX_STEPS'],'NUM_EPOCHS=',os.environ['NUM_EPOCHS'],
      '| lr=',os.environ['LEARNING_RATE'],'->',os.environ['FINAL_LEARNING_RATE'],
      '| eval/save@',os.environ['EVAL_EVERY_STEPS'],'| live_log_require=',os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD'],flush=True)
print('Dataset=micro 979 | INIT=086 pinado | sha pinado (LF) | OUTPUT=',os.environ['OUTPUT_REPO'],flush=True)
print(('>>> ATENCAO: treino REAL (~2-3h). Confirme GO antes. <<<' if MODE=='REAL'
       else '>>> SMOKE: valida tudo + mede tempo/step. Apos OK do Claude, troque MODE=REAL. <<<'), flush=True)

In [ ]:
import subprocess, sys, os, time
os.chdir('/content/kg1')
os.environ['KG1_LIVE_LOG_HF_REPO']='felipesp1983/kg1-live-logs'
os.environ['KG1_LIVE_LOG_HF_REPO_TYPE']='dataset'
os.environ['RUN_ID']='v1244_train_'+time.strftime('%Y%m%d_%H%M%S')
os.environ.setdefault('KG1_REQUIRE_LIVE_LOG_UPLOAD','1')
os.environ.setdefault('KG1_WATCHDOG_STALE_SECONDS','2700')
os.environ.setdefault('KG1_LIVE_LOG_UPLOAD_EVERY','45')
print('LIVE RUN_ID=', os.environ['RUN_ID'], '-> HF:', os.environ['KG1_LIVE_LOG_HF_REPO']+'/colab/'+os.environ['RUN_ID'], flush=True)
print('   adapter ->', os.environ['OUTPUT_REPO']+'/runs/'+os.environ['RUN_ID']+'/{final,checkpoint-N}', flush=True)
r=subprocess.run([sys.executable,'scripts/kg1_colab_realtime_runner.py','--','python','scripts/hf_job_train_v90.py'])
print('RETURN_CODE=', r.returncode, flush=True)
print('Se RC=0 e adapter subiu -> rode NOTEBOOK B (CAND_RUN_ID='+os.environ['RUN_ID']+') p/ ACC real no full947.', flush=True)